# ACC 380K Demo: Classify 10-K Passages with an AI Model

In HW3 you pulled every passage around a phrase out of Tesla's 10-K. An auditor reading those passages still has to decide **what each one is about**. Here we hand that sorting job to an AI model.

The plan:
1. Download Tesla's FY2025 10-K (same as HW3).
2. Pull every passage that mentions **Musk** (same `collect_contexts` idea as HW3).
3. Send each passage to the model with one multiple-choice question.
4. Put the answers in a DataFrame and count them.

The model is **Jev**. It is not a chatbot: you give it a passage and a question with named answer choices, and it returns the best choice plus a probability for each choice. We call it through a class relay that has a spending cap, so nobody needs their own account.

In [ ]:
%pip install -q edgartools==5.58.0

In [ ]:
import edgar
import requests
import pandas as pd

sec_identity = "Your Name your_email@example.com"  # Replace with your own name and email.
edgar.set_identity(sec_identity)

URL = "https://acc380k-jev-relay.nicholashallman.workers.dev"   # instructor gives you this
KEY = "PASTE-CLASS-KEY-HERE"   # instructor gives you this
EID = "your_ut_eid"            # labels your usage; each EID has a small budget

## 1. Download the 10-K

Same accession number as HW3's 2025 filing. `filing.document.markdown()` turns the filing into plain text.

In [ ]:
filing = edgar.get_by_accession_number("0001628280-26-003952")
text = filing.document.markdown()
print(f"{len(text):,} characters")

## 2. Pull the passages

This is HW3's `collect_contexts`: find each appearance of the phrase and keep some text on either side.

In [ ]:
def collect_contexts(text, search_phrase, context_length):
    """Return numbered context passages for every appearance of the phrase."""
    contexts = {}
    search_start = 0
    for appearance in range(1, text.count(search_phrase) + 1):
        position = text.find(search_phrase, search_start)
        start = max(position - context_length, 0)
        stop = position + len(search_phrase) + context_length
        contexts[appearance] = text[start:stop]
        search_start = position + len(search_phrase)
    return contexts

passages = collect_contexts(text, "Musk", 300)
print(f"{len(passages)} passages mention Musk")
print(passages[1])

## 3. Ask the model one question

The question has **instructions** plus named **choices**, each with a short description. Good descriptions matter more than anything else here.

In [ ]:
question = {
    "type": "choice",
    "instructions": "From an auditor's point of view, what is this 10-K passage mainly about with respect to Elon Musk?",
    "criteria": {
        "key_person_risk": "Tesla depends on Musk; losing him or his attention could hurt the business",
        "related_party": "Transactions, loans, or arrangements between Tesla and Musk or companies he controls (SpaceX, xAI, etc.)",
        "compensation_governance": "Musk's pay, stock awards, voting power, board role, or shareholder votes",
        "other": "Anything else, such as a signature line or a list of officers",
    },
}

def classify(passage):
    """Send one passage to the model and return its answer as a dictionary."""
    response = requests.post(
        URL,
        json={"state": passage, "questions": {"label": question}},
        headers={"x-class-key": KEY, "x-student": EID},
        timeout=60,
    )
    data = response.json()
    if response.status_code != 200:
        print("Problem:", response.status_code, data)
        return {"label": None, "confidence": None}
    answer = data["answers"]["label"]
    return {"label": answer["choice"], "confidence": answer["confidence"]}

classify(passages[1])

## 4. Classify a handful of passages

We only send the first 10 to keep the demo quick and cheap.

In [ ]:
rows = []
for number in list(passages)[:10]:
    result = classify(passages[number])
    rows.append({"number": number,
                 "label": result["label"],
                 "confidence": result["confidence"],
                 "passage": passages[number][200:450]})

results = pd.DataFrame(rows)
results

In [ ]:
results["label"].value_counts()

**Auditor's check:** read a few passages yourself. Do you agree with the labels? Which ones have low confidence, and why might the model be unsure? The model sorts; you still judge.

## Your turn

Change the phrase and the choices to ask your own question. Ideas:
- `"tariff"`: does the passage describe a **cost risk**, a **revenue risk**, or **neither**?
- `"warranty"`: is it about an **estimate/reserve**, a **policy description**, or **other**?
- `"material weakness"` or `"impairment"`: is the tone **reassuring**, **concerning**, or **neutral**?

Keep each choice description short and specific. Each EID has a small budget, so classify about 10 passages at a time.

In [ ]:
# Change the phrase, the instructions, and the choices, then run this cell.
passages = collect_contexts(text, "tariff", 300)

question = {
    "type": "choice",
    "instructions": "Write your question here",
    "criteria": {
        "choice_1": "describe choice 1",
        "choice_2": "describe choice 2",
    },
}
